
# CKA Plot from Runs (MAE & FunMMA)

This notebook reads outputs produced by **`cka_dump_meta_single.py`** (each run has `meta.json`, `csv/`, `mats/`)
and generates merged plots from **two runs** (e.g., MAE and FunMMA) or from a **single run**.

**What it makes**
- Layer-wise line plots for snapshot epochs (model by color; token by linestyle/marker)
- Heatmaps: 2×2 grid (if both runs are provided) or 1×2 pair (if single run)
- Epoch-wise trajectory plots (late-layer mean CKA)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

COLOR_BLUE = "#0D5EA6"  # FunMMA (blue)
COLOR_RED = "#E14434"  # MAE (red)

%matplotlib inline

In [ ]:
def plot_layerwise_lines(
    diag_df: pd.DataFrame,
    epoch: int,
    out_dir: Path,
    marker_size: float = 5.0,
    line_width: float = 1.5,
):
    """
    Plot layer-wise CKA diagonal values.
    - Color: model (MAE=red, FunMMA=blue)
    - Linestyle/marker: token (HF=solid+o, LF=dashed+^)
    Requires columns that may include any subset of: mae_hf, mae_lf, fun_hf, fun_lf
    저장: pdf
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    layers = diag_df["layer"].to_numpy()

    plt.figure(figsize=(5, 5))
    plt.rcParams["font.family"] = "Times New Roman"

    if {"mae_hf", "mae_lf"}.issubset(diag_df.columns):
        plt.plot(
            layers,
            diag_df["mae_hf"],
            marker="^",
            linestyle="--",
            label="MAE — High-freq.",
            color=COLOR_BLUE,
            markersize=marker_size,
            linewidth=line_width,
        )
        plt.plot(
            layers,
            diag_df["mae_lf"],
            marker="^",
            linestyle="--",
            label="MAE — Low-freq.",
            color=COLOR_RED,
            markersize=marker_size,
            linewidth=line_width,
        )

    if {"fun_hf", "fun_lf"}.issubset(diag_df.columns):
        plt.plot(
            layers,
            diag_df["fun_hf"],
            marker="o",
            linestyle="-",
            label="FunMMA — High-freq.",
            color=COLOR_BLUE,
            markersize=marker_size,
            linewidth=line_width,
        )
        plt.plot(
            layers,
            diag_df["fun_lf"],
            marker="o",
            linestyle="-",
            label="FunMMA — Low-freq.",
            color=COLOR_RED,
            markersize=marker_size,
            linewidth=line_width,
        )

    plt.xlabel("Layer index", fontname="Times New Roman", fontsize=18)
    plt.ylabel("CKA Similarity", fontname="Times New Roman", fontsize=18)
    plt.ylim(0.1, 1.02)
    plt.grid(alpha=0.25)

    # x축 layer index 정수로, ticks 안쪽으로
    layer_min = int(np.min(layers))
    layer_max = int(np.max(layers))
    xticks = np.arange(layer_min, layer_max + 1)
    plt.xticks(xticks, fontsize=10)
    plt.yticks(fontsize=10)
    plt.tick_params(axis="both", labelsize=14, direction="in")  # ticks 안쪽으로

    plt.legend(
        ncol=1, loc="best", prop={"family": "Times New Roman", "size": 16}
    )  # 레전드 세로로 배치
    out = out_dir / f"layerwise_lines_epoch{epoch:03d}.pdf"
    plt.tight_layout()
    plt.savefig(out, dpi=220)
    plt.show()
    plt.close()


def plot_epochwise_trajectory(
    df_traj: pd.DataFrame, out_dir: Path, marker_size: float = 5.0, line_width: float = 1.5
):
    """
    Plot epoch-wise trajectory (late-layer mean CKA).
    - Color: model (MAE=red, FunMMA=blue)
    - Linestyle/marker: token (HF=solid+o, LF=dashed+^)
    Requires columns that may include any subset of: mae_hf, mae_lf, fun_hf, fun_lf
    저장: pdf
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    epochs = df_traj["epoch"].to_numpy() + 1

    plt.figure(figsize=(5, 5))
    plt.rcParams["font.family"] = "Times New Roman"

    if {"mae_hf", "mae_lf"}.issubset(df_traj.columns):
        plt.plot(
            epochs,
            df_traj["mae_hf"],
            marker="^",
            linestyle="--",
            label="MAE — High-freq.",
            color=COLOR_BLUE,
            markersize=marker_size,
            linewidth=line_width,
        )
        plt.plot(
            epochs,
            df_traj["mae_lf"],
            marker="^",
            linestyle="--",
            label="MAE — Low-freq.",
            color=COLOR_RED,
            markersize=marker_size,
            linewidth=line_width,
        )

    if {"fun_hf", "fun_lf"}.issubset(df_traj.columns):
        plt.plot(
            epochs,
            df_traj["fun_hf"],
            marker="o",
            linestyle="-",
            label="FunMMA — High-freq.",
            color=COLOR_BLUE,
            markersize=marker_size,
            linewidth=line_width,
        )
        plt.plot(
            epochs,
            df_traj["fun_lf"],
            marker="o",
            linestyle="-",
            label="FunMMA — Low-freq.",
            color=COLOR_RED,
            markersize=marker_size,
            linewidth=line_width,
        )

    plt.xlabel("Epoch", fontname="Times New Roman", fontsize=18)
    plt.ylabel("CKA Similarity", fontname="Times New Roman", fontsize=18)
    plt.ylim(0.1, 1.02)
    plt.grid(alpha=0.25)

    # x축 epoch 정수로, ticks 안쪽으로
    plt.yticks(fontsize=10)
    plt.tick_params(axis="both", labelsize=14, direction="in")  # ticks 안쪽으로

    plt.legend(
        ncol=1, loc="best", prop={"family": "Times New Roman", "size": 16}
    )  # 레전드 세로로 배치
    out = out_dir / "epochwise_trajectory.pdf"
    plt.tight_layout()
    plt.savefig(out, dpi=220)
    plt.show()
    plt.close()

In [ ]:
def plot_heatmap_pair(mats_dir: Path, epoch: int, model_name: str, prefix: str, out_dir: Path):
    """
    Show a 1x2 heatmap pair (HF/LF) for a single model run directory.
    Expects files: epochXXX_{prefix}_hf.npz and epochXXX_{prefix}_lf.npz
    저장: pdf
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    p_hf = mats_dir / f"epoch{epoch:03d}_{prefix}_hf.npz"
    p_lf = mats_dir / f"epoch{epoch:03d}_{prefix}_lf.npz"
    if not (p_hf.exists() and p_lf.exists()):
        print(f"[warn] heatmap npz not found for epoch {epoch} in {mats_dir}; skip.")
        return
    M_hf = np.load(p_hf)["M"]
    M_lf = np.load(p_lf)["M"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    vmin, vmax, cmap = 0.0, 1.0, "magma"
    im0 = axes[0].imshow(M_hf, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{model_name}: HF ~ Full")
    axes[1].imshow(M_lf, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title(f"{model_name}: LF ~ Full")
    for ax in axes.ravel():
        ax.set_xlabel("Layer (token-b)")
        ax.set_ylabel("Layer (token-a)")
    cbar_ax = fig.add_axes([0.92, 0.2, 0.015, 0.6])
    fig.colorbar(im0, cax=cbar_ax)
    fig.suptitle(f"CKA Heatmaps — Epoch {epoch}", y=0.98)
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    out = out_dir / f"heatmaps_{prefix}_epoch{epoch:03d}.pdf"
    plt.savefig(out, dpi=220)
    plt.show()
    plt.close()


def plot_heatmap_grid_dual(
    mae_mats: Path, fun_mats: Path, epoch: int, out_dir: Path, prefix_mae: str, prefix_fun: str
):
    """
    2x2 heatmap (MAE/Fun × HF/LF) using npz files from two different run dirs.
    Expects:
      mae_mats/epochXXX_mae_hf.npz, mae_mats/epochXXX_mae_lf.npz
      fun_mats/epochXXX_fun_hf.npz, fun_mats/epochXXX_fun_lf.npz
    저장: pdf
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    reqs = {
        "mae_hf": mae_mats / f"epoch{epoch:03d}_{prefix_mae}_hf.npz",
        "mae_lf": mae_mats / f"epoch{epoch:03d}_{prefix_mae}_lf.npz",
        "fun_hf": fun_mats / f"epoch{epoch:03d}_{prefix_fun}_hf.npz",
        "fun_lf": fun_mats / f"epoch{epoch:03d}_{prefix_fun}_lf.npz",
    }
    if not all(p.exists() for p in reqs.values()):
        print(f"[warn] not all dual npz found for epoch {epoch}; skip grid.")
        return

    M_mae_hf = np.load(reqs["mae_hf"])["M"]
    M_mae_lf = np.load(reqs["mae_lf"])["M"]
    M_fun_hf = np.load(reqs["fun_hf"])["M"]
    M_fun_lf = np.load(reqs["fun_lf"])["M"]

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    vmin, vmax, cmap = 0.0, 1.0, "magma"
    im = axes[0, 0].imshow(M_mae_hf, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, 0].set_title("MAE: HF ~ Full")
    axes[0, 1].imshow(M_fun_hf, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, 1].set_title("FunMMA: HF ~ Full")
    axes[1, 0].imshow(M_mae_lf, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1, 0].set_title("MAE: LF ~ Full")
    axes[1, 1].imshow(M_fun_lf, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1, 1].set_title("FunMMA: LF ~ Full")
    for ax in axes.ravel():
        ax.set_xlabel("Layer (token-b)")
        ax.set_ylabel("Layer (token-a)")
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cbar_ax)
    fig.suptitle(f"CKA Heatmaps — Epoch {epoch}", y=0.98)
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    out = out_dir / f"heatmaps_dual_epoch{epoch:03d}.pdf"
    plt.savefig(out, dpi=220)
    plt.show()
    plt.close()

In [ ]:
def _load_run_meta(run_or_meta: Path) -> dict:
    """Accept run directory (with meta.json) or path to meta.json."""
    run_or_meta = Path(run_or_meta)
    meta_path = run_or_meta / "meta.json" if run_or_meta.is_dir() else run_or_meta
    if not meta_path.exists():
        raise FileNotFoundError(f"meta.json not found at: {meta_path}")
    with open(meta_path) as f:
        meta = json.load(f)
    return meta


def _rename_diag(df: pd.DataFrame, prefix: str, as_model: str) -> pd.DataFrame:
    """Rename '<prefix>_hf/lf' -> '<as_model>_hf/lf' (as_model in {'mae','fun'})."""
    cols = {}
    if f"{prefix}_hf" in df.columns:
        cols[f"{prefix}_hf"] = f"{as_model}_hf"
    if f"{prefix}_lf" in df.columns:
        cols[f"{prefix}_lf"] = f"{as_model}_lf"
    return df.rename(columns=cols)


## Parameters

- `run_fun`: path to **FunMMA run** directory or its `meta.json`  
- `run_mae`: (optional) path to **MAE run** directory or its `meta.json`  
- `out_dir`: where to save plots (default: sibling `plots/` near runs)  
- `snapshot_epochs_override`: list of epochs to plot (if `None`, auto from meta)  
- `make_heatmaps`: whether to render heatmaps  
- style: `marker_size`, `line_width`


In [ ]:
# === User parameters ===
run_fun = "your_path/src/tasks/pretrain/experiment/cka_res/run_20250902-134951"  # e.g., "/path/to/fun/run_2025..."
run_mae = "your_path/src/tasks/pretrain/experiment/cka_res/run_20250902-134942"  # e.g., "/path/to/mae/run_2025..." (optional; leave empty if none)
out_dir = "your_path/src/tasks/pretrain/experiment/cka_res/plots"  # e.g., "/path/to/output/plots" (optional; leave empty for default)

snapshot_epochs_override = None  # e.g., [19, 49, 99]
make_heatmaps = True

marker_size = 6
line_width = 1.5

In [ ]:
# === Main execution ===
from pathlib import Path

# Load metas
if not run_fun:
    raise ValueError("Please set 'run_fun' to a valid run dir or its meta.json path.")
meta_fun = _load_run_meta(Path(run_fun))

meta_mae = None
if run_mae:
    meta_mae = _load_run_meta(Path(run_mae))

# Paths & names
csv_fun = Path(meta_fun["paths"]["csv"])
mats_fun = Path(meta_fun["paths"]["mats"])
prefix_fun = meta_fun.get("prefix", "fun")
name_fun = meta_fun.get("model_name", "FunMMA")

csv_mae = mats_mae = prefix_mae = name_mae = None
if meta_mae:
    csv_mae = Path(meta_mae["paths"]["csv"])
    mats_mae = Path(meta_mae["paths"]["mats"])
    prefix_mae = meta_mae.get("prefix", "mae")
    name_mae = meta_mae.get("model_name", "MAE")

# Out dir
if out_dir:
    out_root = Path(out_dir)
else:
    out_root = Path(csv_fun).resolve().parents[1] / "plots"
out_snap = out_root / "snapshots"
out_heat = out_root / "heatmaps"
out_snap.mkdir(parents=True, exist_ok=True)
out_heat.mkdir(parents=True, exist_ok=True)

# Snapshot epochs
if snapshot_epochs_override is not None:
    snapshot_epochs = list(snapshot_epochs_override)
else:
    snaps_fun = set(meta_fun.get("snapshot_epochs", []))
    if meta_mae:
        snaps_mae = set(meta_mae.get("snapshot_epochs", []))
        inter = sorted(snaps_fun & snaps_mae)
        snapshot_epochs = inter if inter else sorted(snaps_fun)
    else:
        snapshot_epochs = sorted(snaps_fun)

print("[info] fun csv:", csv_fun)
if meta_mae:
    print("[info] mae csv:", csv_mae)
print("[info] snapshot_epochs:", snapshot_epochs)
print("[info] output:", out_root)

# 1) Layer-wise snapshot lines (+ heatmaps)
for e in snapshot_epochs:
    fun_diag = csv_fun / f"diag_epoch{e:03d}.csv"
    if not fun_diag.exists():
        print(f"[warn] missing FUN diag csv for epoch {e}: {fun_diag}")
        continue
    df_fun = pd.read_csv(fun_diag)
    df_fun = _rename_diag(df_fun, prefix_fun, "fun")

    if meta_mae:
        mae_diag = csv_mae / f"diag_epoch{e:03d}.csv"
        if mae_diag.exists():
            df_mae = pd.read_csv(mae_diag)
            df_mae = _rename_diag(df_mae, prefix_mae, "mae")
            df_merged = pd.merge(df_fun, df_mae, on="layer", how="outer")
        else:
            print(f"[warn] missing MAE diag csv for epoch {e}: {mae_diag}")
            df_merged = df_fun
    else:
        df_merged = df_fun

    plot_layerwise_lines(
        df_merged, epoch=e, out_dir=out_snap, marker_size=marker_size, line_width=line_width
    )

    # Heatmaps
    if make_heatmaps:
        if meta_mae:
            plot_heatmap_grid_dual(
                mats_mae,
                mats_fun,
                epoch=e,
                out_dir=out_heat,
                prefix_mae=prefix_mae,
                prefix_fun=prefix_fun,
            )
        else:
            plot_heatmap_pair(
                mats_fun, epoch=e, model_name=name_fun, prefix=prefix_fun, out_dir=out_heat
            )

# 2) Epoch-wise trajectory
fun_traj = csv_fun / "epochwise_trajectory.csv"
if fun_traj.exists():
    df_fun_t = pd.read_csv(fun_traj).rename(
        columns={f"{prefix_fun}_hf": "fun_hf", f"{prefix_fun}_lf": "fun_lf"}
    )
    if meta_mae:
        mae_traj = csv_mae / "epochwise_trajectory.csv"
        if mae_traj.exists():
            df_mae_t = pd.read_csv(mae_traj).rename(
                columns={f"{prefix_mae}_hf": "mae_hf", f"{prefix_mae}_lf": "mae_lf"}
            )
            df_t = pd.merge(df_fun_t, df_mae_t, on="epoch", how="outer")
        else:
            print(f"[warn] missing MAE trajectory CSV: {mae_traj}")
            df_t = df_fun_t
    else:
        df_t = df_fun_t

    cols = ["epoch"] + [c for c in ["mae_hf", "mae_lf", "fun_hf", "fun_lf"] if c in df_t.columns]
    df_t = df_t[cols]
    plot_epochwise_trajectory(
        df_t, out_dir=out_root, marker_size=marker_size, line_width=line_width
    )
else:
    print(f"[warn] missing FUN trajectory CSV: {fun_traj}")

print(f"[OK] Plots saved under: {out_root.resolve()}")